In [ ]:
import pandas as pd

books = pd.read_csv("../data/books_with_categories.csv")

In [ ]:
from transformers import pipeline

# classifier for Eckman 6: anger, disgust, fear, joy, neutral, sadness, surprise
classifier = pipeline("text-classification",
                      model="j-hartmann/emotion-english-distilroberta-base",
                      top_k = None,
                      device = "mps")
classifier("I love this!")

In [ ]:
books["description"][0]

In [ ]:
classifier(books["description"][0])
# not a very accurate classification

In [ ]:
# classify sentences individually
classifier(books["description"][0].split("."))

In [ ]:
sentences = books["description"][0].split(".")
predictions = classifier(sentences)
sorted(predictions[0], key=lambda x: x["label"])
# take the sentence with the highest probability for each sentiment

In [ ]:
import numpy as np

emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
isbn = []
emotion_scores = {label: [] for label in emotion_labels}

# creates a dictionary for each description containing the maximumm probability for each emotion
def calculate_max_emotion_scores(predictions):
    per_emotion_scores = {label: [] for label in emotion_labels}
    for prediction in predictions:
        sorted_predictions = sorted(prediction, key=lambda x: x["label"])
        for index, label in enumerate(emotion_labels):
            per_emotion_scores[label].append(sorted_predictions[index]["score"])
    return {label: np.max(scores) for label, scores in per_emotion_scores.items()}

In [ ]:
# test for the first 10 books
for i in range(10):
    isbn.append(books["isbn13"][i])
    sentences = books["description"][i].split(".")
    predictions = classifier(sentences)
    max_scores = calculate_max_emotion_scores(predictions)
    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])

In [ ]:
emotion_scores

In [ ]:
from tqdm import tqdm
import re

emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
isbn = []
emotion_scores = {label: [] for label in emotion_labels}

for i in tqdm(range(len(books))):
    isbn.append(books["isbn13"][i])
    sentences = re.split(r'[.?!]+', books["description"][i])
    predictions = classifier(sentences)
    max_scores = calculate_max_emotion_scores(predictions)
    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])

In [ ]:
emotions_df = pd.DataFrame(emotion_scores)
emotions_df["isbn13"] = isbn

In [ ]:
emotions_df

In [ ]:
# add the maximum score for each emotion to the book vectors
books = pd.merge(books, emotions_df, on = "isbn13")

In [ ]:
books

In [ ]:
books.to_csv("../data/books_with_emotions.csv", index = False)